## Ingest Fact Data into Bronze Layer

In [ ]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

In [ ]:
catalog_name = "ecommerce"
raw_data_path = '/Volumes/ecommerce/source_data/raw'

### Order Dataframe & Table

In [ ]:
# Define schema struture..
order_item_schema = StructType([
    StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",        StringType(), True),
])

# prepared dataframe..
order_item_df = spark.read.option("header", True).option("delimeter", ",").schema(order_item_schema).csv(f"{raw_data_path}/order_items/landing/*.csv")

# add some extra columns..
order_item_df = order_item_df.withColumn("created_at", F.current_timestamp()) \
                    .withColumn("source_file", F.col("_metadata.file_path"))

display(order_item_df.limit(2))

# convert df to table.
order_item_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.broze.brz_order_items")

In [ ]:
%%sql
SELECT COUNT(*) FROM ecommerce.broze.brz_order_items;